In [ ]:
import json
import math
import time
import chromadb
from chromadb.utils import embedding_functions
from datasets import load_dataset
from tqdm import tqdm

In [ ]:
# ==========================================
# 1. METRICS ENGINE
# ==========================================


def calculate_metrics(retrieved_ids, target_id, latency_ms):
    if target_id not in retrieved_ids:
        return {"recall": 0, "mrr": 0.0, "ndcg": 0.0, "latency_ms": latency_ms}
    rank = retrieved_ids.index(target_id) + 1
    return {
        "recall": 1,
        "mrr": 1.0 / rank,
        "ndcg": 1.0 / math.log2(rank + 1),
        "latency_ms": latency_ms,
    }


def print_scorecard(model_name, dataset_name, metrics_list):
    total = len(metrics_list)
    if total == 0:
        return
    avg_recall = sum(m["recall"] for m in metrics_list) / total
    avg_mrr = sum(m["mrr"] for m in metrics_list) / total
    avg_ndcg = sum(m["ndcg"] for m in metrics_list) / total
    avg_latency = sum(m["latency_ms"] for m in metrics_list) / total

    print("\n" + "-" * 50)
    print(f"📂 DATASET: {dataset_name}")
    print(f"Recall@10:         {avg_recall:.4f} ({(avg_recall*100):.1f}%)")
    print(f"MRR:               {avg_mrr:.4f}")
    print(f"nDCG@10:           {avg_ndcg:.4f}")
    print(f"Avg Query Latency: {avg_latency:.2f} ms")
    print("-" * 50)

In [ ]:
# ==========================================
# 2. LOAD BOTH DATASETS INTO MEMORY
# ==========================================
print("Loading datasets into memory...")

# A. Synthetic Data
with open("../data/synthetic_ground_truth_100.json", "r") as f:
    synthetic_ground_truth = json.load(f)
arxiv_abstracts = load_dataset("gfissore/arxiv-abstracts-2021", split="train")
synthetic_docs = arxiv_abstracts.select(range(100))

In [ ]:
# B. c1khoa Benchmark Data
benchmark_data = load_dataset("c1khoa/papers_retrieval_arxiv", split="train")
benchmark_unique_docs = {}
benchmark_ground_truth = []

In [ ]:
for row in benchmark_data:
    doc_id = str(row.get("paper_id", row.get("id")))
    doc_text = row.get("abstract", row.get("document", ""))
    query_text = row.get("query", "")

    if doc_id not in benchmark_unique_docs:
        benchmark_unique_docs[doc_id] = doc_text

    benchmark_ground_truth.append({"query": query_text, "correct_paper_id": doc_id})

benchmark_doc_ids = list(benchmark_unique_docs.keys())
benchmark_doc_texts = list(benchmark_unique_docs.values())

In [ ]:
# ==========================================
# 3. THE MASTER EVALUATION LOOP
# ==========================================
MODELS_TO_TEST = [
    "all-MiniLM-L6-v2",
    "BAAI/bge-small-en-v1.5",
    "BAAI/bge-large-en-v1.5",
    "nomic-ai/nomic-embed-text-v1.5",
]

chroma_client = chromadb.Client()

for model_name in MODELS_TO_TEST:
    print("\n" + "=" * 50)
    print(f"🚀 TESTING MODEL: {model_name}")
    print("=" * 50)

    try:
        if "nomic" in model_name:
            emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
                model_name=model_name, trust_remote_code=True
            )
        else:
            emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
                model_name=model_name
            )

        # -----------------------------------------
        # TEST 1: Synthetic Dataset
        # -----------------------------------------
        col_synth_name = f"synth_{model_name.replace('/', '-').replace('.', '-')}"
        try:
            chroma_client.delete_collection(name=col_synth_name)
        except Exception:
            pass

        col_synth = chroma_client.create_collection(
            name=col_synth_name,
            embedding_function=emb_fn,
            metadata={"hnsw:space": "cosine"},
        )
        col_synth.add(
            documents=[p["abstract"] for p in synthetic_docs],
            ids=[p["id"] for p in synthetic_docs],
        )

        synth_metrics = []
        for item in tqdm(
            synthetic_ground_truth, desc=f"[{model_name}] Synthetic Queries"
        ):
            start = time.perf_counter()
            res = col_synth.query(query_texts=[item["query"]], n_results=10)
            latency = (time.perf_counter() - start) * 1000
            synth_metrics.append(
                calculate_metrics(res["ids"][0], item["correct_paper_id"], latency)
            )

        print_scorecard(model_name, "Local Synthetic (100 papers)", synth_metrics)

        # -----------------------------------------
        # TEST 2: c1khoa Benchmark
        # -----------------------------------------
        col_bench_name = f"bench_{model_name.replace('/', '-').replace('.', '-')}"
        try:
            chroma_client.delete_collection(name=col_bench_name)
        except Exception:
            pass

        col_bench = chroma_client.create_collection(
            name=col_bench_name,
            embedding_function=emb_fn,
            metadata={"hnsw:space": "cosine"},
        )

        # Batch insert for benchmark to handle memory safely
        for i in range(0, len(benchmark_doc_ids), 500):
            col_bench.add(
                documents=benchmark_doc_texts[i : i + 500],
                ids=benchmark_doc_ids[i : i + 500],
            )

        bench_metrics = []
        for item in tqdm(benchmark_ground_truth, desc=f"[{model_name}] c1khoa Queries"):
            start = time.perf_counter()
            res = col_bench.query(query_texts=[item["query"]], n_results=10)
            latency = (time.perf_counter() - start) * 1000
            bench_metrics.append(
                calculate_metrics(res["ids"][0], item["correct_paper_id"], latency)
            )

        print_scorecard(model_name, "c1khoa Benchmark", bench_metrics)

    except Exception as e:
        print(f"❌ FAILED to evaluate {model_name}. Error: {e}")